In [ ]:
import pandas as _hex_pandas
import datetime as _hex_datetime
import json as _hex_json

In [ ]:
hex_scheduled = _hex_json.loads("false")

In [ ]:
hex_user_email = _hex_json.loads("\"example-user@example.com\"")

In [ ]:
hex_user_attributes = _hex_json.loads("{}")

In [ ]:
hex_run_context = _hex_json.loads("\"logic\"")

In [ ]:
hex_timezone = _hex_json.loads("\"UTC\"")

In [ ]:
hex_project_id = _hex_json.loads("\"019d5b49-c6ca-7dda-9e70-4e9230585df3\"")

In [ ]:
hex_project_name = _hex_json.loads("\"Production Plan (linear programming)\"")

In [ ]:
hex_status = _hex_json.loads("\"Completed\"")

In [ ]:
hex_categories = _hex_json.loads("[\"Ops\",\"Product\"]")

In [ ]:
hex_color_palette = _hex_json.loads("[\"#4C78A8\",\"#F58518\",\"#E45756\",\"#72B7B2\",\"#54A24B\",\"#EECA3B\",\"#B279A2\",\"#FF9DA6\",\"#9D755D\",\"#BAB0AC\"]")

In [ ]:
!uv pip install pulp -q
import pandas as pd
import pulp

Resources available



In [ ]:
product = pd.read_csv('production_planning.product.csv') # dataframe with product data
resource = pd.read_csv('production_planning.resource.csv') # dataframe with resource data

In [ ]:
resource

,resource,available_hours,hourly_cost
0,machine_time,585,300
1,assembly_time,675,240
2,finishing_time,1110,180


Production Costs & Demand



In [ ]:
product

,parameter,P1,P2,P3,P4,P5
0,demand,7000,6000,4000,5000,1000
1,cost_inhouse,93,109,44,80,46
2,cost_outsource,104,123,61,90,59
3,machine_time,2,1,2,2,1
4,assembly_time,1,3,3,1,3
5,finishing_time,3,4,4,2,3


Parameters of the LP model



In [ ]:
PRODUCTS = list(product)[1:] # list of products (all but first column header)
N = len(PRODUCTS) # number of products
D, C, P, m, a, f = product.values[:,1:].tolist() # parameter values

RESOURCES = resource.resource.tolist() # list of resources
Tm, Ta, Tf = (60*resource.available_hours).tolist() # parameter values
HOURLY_COST = resource.hourly_cost.tolist()

In [ ]:
LP_file = 'production_planning.lp' # name of LP model file
from pulp import LpProblem, LpMinimize
prob = LpProblem(LP_file, LpMinimize) # Create LP model object

Defining the decision variables



In [ ]:
from pulp import LpVariable
X = [LpVariable(f'x_{i+1}',0) for i in range(N)] # quantities produced
Y = [LpVariable(f'y_{i+1}',0) for i in range(N)] # quantities purchased

Objective Function



In [ ]:
from pulp import lpSum
prob += lpSum(C[i]*X[i] + P[i]*Y[i] for i in range(N)) # objective function

/ipython/.venv/lib/python3.12/site-packages/pulp/pulp.py:1865: UserWarning: Overwriting previously set objective.
  warnings.warn("Overwriting previously set objective.")


Demand constraints



In [ ]:
for i in range(N):
    prob += X[i] + Y[i] >= D[i] , 'Demand for '+ PRODUCTS[i]

Resource Constraints



In [ ]:
for idx, (r, t, q) in enumerate(zip(RESOURCES, [m, a, f], [Tm, Ta, Tf])): # for each resource
    prob += sum(t[i]*X[i] for i in range(N)) <= q, f"{r}_availability_{idx}" # add unique resource availability constraint

LP Model save and solve



In [ ]:
prob.writeLP(LP_file) # write model to LP file
print(open(LP_file).read()) # show LP model

In [ ]:
from pulp import LpProblem, LpMinimize, LpVariable, value, LpStatus
prob.solve() # solve problem
status = LpStatus[prob.status] # optimal found?
print("Solution: ", status)
min_cost = value(prob.objective) # objective function value
print(f'Minimum cost = $ {min_cost:,.2f}') # show optimal cost

Optimal Decision variables



In [ ]:
X_opt = [x.varValue for x in X] # optimal quantities produced
Y_opt = [y.varValue for y in Y] # optimal quantities purchased

# Save and display optimal quantities in a dataframe
opt_qty = pd.DataFrame([X_opt, Y_opt], columns=PRODUCTS).round(2) # convert to dataframe
opt_qty.insert(0, 'Quantity', ['Produced', 'Purchased']) # add first column
opt_qty.to_csv("optimal_quantities.csv", index=False) # save results
print('Optimal quantities:')
opt_qty # display optimal quantities

Solutions displayed



In [ ]:
# import jinja2
# raw_query = """
#     SELECT
#         Quantity AS type,
#         product,
#         quantity_value
#     FROM opt_qty
#     UNPIVOT (quantity_value FOR product IN (P1, P2, P3, P4, P5))
#     
# """
# sql_query = jinja2.Template(raw_query).render(vars())

Resources Used



In [ ]:
resource_use = [] # list with results
for r in RESOURCES: # for each resource
    # Constraint names may not match directly; search for a key containing the resource
    try:
        name = next(k for k in prob.constraints.keys() if r in k)
        c = prob.constraints[name] # details for constraint
        available = -c.constant # available minutes (RHS of the constraint)
        unused = c.slack # minutes not used
        used = available - unused # minutes used
        resource_use.append([r, used, available, unused]) # append results for resource
    except StopIteration:
        resource_use.append([r, None, None, None]) # Could not find constraint
# convert results to dataframe
resource_use = pd.DataFrame(resource_use, columns = ['resource', 'used', 'available', 'unused'])
resource_use.to_csv('resource_use.csv', index=False) # sane results
resource_use # show results

Sensitivity Analysis



In [ ]:
max_price = [] # results with maximum cost per additional hour
for r, hc in zip(RESOURCES, HOURLY_COST): # for each resource and its hourly cost
    # Find constraint by matching resource name
    found = False
    for cname in prob.constraints:
        if (cname == f'{r}_availability') or (cname == r):
            c = prob.constraints[cname]
            found = True
            break
    if not found:
        max_price.append([r, hc, float('nan'), float('nan')])
        continue
    shadow_price = -c.pi # cost savings per minute of resource
    savings = 60*shadow_price # cost savings per additional hour of resource
    price = hc + savings # maximum price for an additional hour of resource
    max_price.append([r, hc, savings, price])
# convert results to dataframe
max_price = pd.DataFrame(max_price, columns = ['resource', 'cost per hour', 'savings per hour', 'max price'])
max_price.to_csv('max_price.csv', index=False) # save results
max_price # show results

In [ ]:
cost_breakdown = pd.DataFrame({
    'product': PRODUCTS,
    'production_cost': [C[i] * X[i].varValue for i in range(N)],
    'purchase_cost': [P[i] * Y[i].varValue for i in range(N)],
    'unit_prod_cost': C,
    'unit_purch_cost': P,
})
cost_breakdown['total_cost'] = cost_breakdown['production_cost'] + cost_breakdown['purchase_cost']

resource_util = resource_use.copy()
resource_util['utilization_pct'] = (resource_util['used'] / resource_util['available'] * 100).round(1)


In [ ]:
# import jinja2
# raw_query = """
#     SELECT resource, 'Used' AS status, used AS minutes FROM resource_util
#     UNION ALL
#     SELECT resource, 'Unused' AS status, unused AS minutes FROM resource_util
#     ORDER BY resource
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
# import jinja2
# raw_query = """
#     SELECT
#         product,
#         'Production' AS cost_type,
#         production_cost AS cost
#     FROM cost_breakdown
#     UNION ALL
#     SELECT
#         product,
#         'Purchase' AS cost_type,
#         purchase_cost AS cost
#     FROM cost_breakdown
#     ORDER BY product, cost_type
#     
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
bottleneck_data = max_price.merge(resource_util[['resource', 'utilization_pct']], on='resource')
bottleneck_data['utilization_pct'] = bottleneck_data['utilization_pct'] / 100
bottleneck_data